Script for parsing CWE XML to raw dataset

In [ ]:

from textwrap import dedent


"""
Extract selected fields from CWE XML (cwec_v4.18.xml) into JSONL + CSV summary.

Outputs:
 - cleaned data/cwe_extracted.jsonl  (one JSON object per Weakness)
 - cleaned data/cwe_summary.csv     (flattened summary for quick view)

Fields extracted (per user request):
 - cwe_id (e.g. "cwe-190")
 - title (Name attribute)
 - description (Description element text)
 - mode_uri (XPath-like location for Modes_Of_Introduction)
 - modeOfIntroductionNote (notes under Introduction/Note)
 - modeOfIntroductionPhase (Introduction/Phase values; list)
 - consequences_uri (XPath-like location for Common_Consequences)
 - consequenceScope (list of Scope values)
 - consequenceImpact (list of Impact values)
 - consequenceNote (list of Note values per consequence)
 - detection_uri (XPath-like location for Detection_Methods)
 - detectionMethod (list of Method values)
 - detectionDescription (list of Description HTML/text)
 - detectionEffectiveness (list of Effectiveness values)
 - detectionEffectivenessNote (list of Effectiveness_Notes values)
 - mitigation_uri (XPath-like location for Potential_Mitigations)
 - mitigationPhase (list of Phase values)
 - mitigationEffectiveness (list of Effectiveness values)
 - mitigationDescription (list of Description text)
 - related_weaknesses (list of dicts: {cwe_id, nature, view_id})

Notes:
 - The script handles the CWE XML default namespace.
 - Outputs go to 'cleaned data/' relative to the script location.
"""

import json
from pathlib import Path
import xml.etree.ElementTree as ET
import csv

INPUT = Path("data/cwec_v4.18.xml")
OUT_DIR = Path("cleaned data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
JSONL_OUT = OUT_DIR / "cwe_extracted.jsonl"
CSV_OUT = OUT_DIR / "cwe_summary.csv"

if not INPUT.exists():
    raise SystemExit(f"Input file not found: {INPUT.resolve()} (place cwec_v4.18.xml in data/)")

# parse with namespace support
tree = ET.parse(INPUT)
root = tree.getroot()

# detect default namespace (if any)
ns = {}
if root.tag.startswith("{"):
    uri = root.tag.split("}")[0].strip("{")
    ns['cwe'] = uri
else:
    ns['cwe'] = ''

def q(tag):
    # qualify tag with namespace prefix 'cwe'
    if ns['cwe']:
        return f"{{{ns['cwe']}}}{tag}"
    return tag

def text_of(elem):
    if elem is None:
        return ""
    # join all text parts inside element (handles nested xhtml blocks)
    return "".join(elem.itertext()).strip()

def gather_multiple_text(parent, path):
    items = []
    for el in parent.findall(path):
        items.append(text_of(el))
    return items

out_items = []
# iterate Weakness elements
for w in root.findall('.//' + q('Weakness')):
    wid = w.get('ID') or ""
    cwe_id = f"cwe-{wid}" if wid else ""
    title = w.get('Name') or ""
    desc = text_of(w.find(q('Description')))

    # Modes of introduction
    modes_parent = w.find(q('Modes_Of_Introduction'))
    mode_uri = ""
    mode_phases = []
    mode_notes = []
    if modes_parent is not None:
        mode_uri = f"/Weaknesses/Weakness[@ID='{wid}']/Modes_Of_Introduction/Introduction"
        for intro in modes_parent.findall(q('Introduction')):
            ph = intro.find(q('Phase'))
            nm = intro.find(q('Note'))
            if ph is not None and ph.text:
                mode_phases.append(ph.text.strip())
            if nm is not None:
                mode_notes.append(text_of(nm))

    # Common consequences
    cons_parent = w.find(q('Common_Consequences'))
    consequences_uri = ""
    consequence_scopes = []
    consequence_impacts = []
    consequence_notes = []
    if cons_parent is not None:
        consequences_uri = f"/Weaknesses/Weakness[@ID='{wid}']/Common_Consequences/Consequence"
        for cons in cons_parent.findall(q('Consequence')):
            scopes = [el.text.strip() for el in cons.findall(q('Scope')) if el.text]
            impacts = [el.text.strip() for el in cons.findall(q('Impact')) if el.text]
            notes = [text_of(el) for el in cons.findall(q('Note')) if el is not None]
            consequence_scopes.append(scopes)
            consequence_impacts.append(impacts)
            consequence_notes.append(notes)

    # Detection methods
    det_parent = w.find(q('Detection_Methods'))
    detection_uri = ""
    detection_methods = []
    detection_descriptions = []
    detection_effectiveness = []
    detection_effectiveness_notes = []
    if det_parent is not None:
        detection_uri = f"/Weaknesses/Weakness[@ID='{wid}']/Detection_Methods/Detection_Method"
        for dm in det_parent.findall(q('Detection_Method')):
            m = dm.find(q('Method'))
            d = dm.find(q('Description'))
            e = dm.find(q('Effectiveness'))
            en = dm.find(q('Effectiveness_Notes'))
            if m is not None:
                detection_methods.append(text_of(m))
            detection_descriptions.append(text_of(d) if d is not None else "")
            detection_effectiveness.append(text_of(e) if e is not None else "")
            detection_effectiveness_notes.append(text_of(en) if en is not None else "")

    # Potential mitigations
    mit_parent = w.find(q('Potential_Mitigations'))
    mitigation_uri = ""
    mitigation_phases = []
    mitigation_effectiveness = []
    mitigation_descriptions = []
    if mit_parent is not None:
        mitigation_uri = f"/Weaknesses/Weakness[@ID='{wid}']/Potential_Mitigations/Mitigation"
        for mit in mit_parent.findall(q('Mitigation')):
            ph = mit.find(q('Phase'))
            desc_el = mit.find(q('Description'))
            eff = mit.find(q('Effectiveness'))
            if ph is not None and ph.text:
                mitigation_phases.append(ph.text.strip())
            mitigation_descriptions.append(text_of(desc_el) if desc_el is not None else "")
            mitigation_effectiveness.append(text_of(eff) if eff is not None else "")

    # Related weaknesses
    related = []
    related_parent = w.find(q('Related_Weaknesses'))
    if related_parent is not None:
        for rw in related_parent.findall(q('Related_Weakness')):
            r_cwe = rw.get('CWE_ID') or ""
            nature = rw.get('Nature') or ""
            view_id = rw.get('View_ID') or ""
            related.append({"cwe_id": f"cwe-{r_cwe}" if r_cwe else "", "nature": nature, "view_id": view_id})

    item = {
        "cwe_id": cwe_id,
        "title": title,
        "description": desc,
        "mode_uri": mode_uri,
        "modeOfIntroductionPhase": mode_phases,
        "modeOfIntroductionNote": mode_notes,
        "consequences_uri": consequences_uri,
        "consequenceScope": consequence_scopes,
        "consequenceImpact": consequence_impacts,
        "consequenceNote": consequence_notes,
        "detection_uri": detection_uri,
        "detectionMethod": detection_methods,
        "detectionDescription": detection_descriptions,
        "detectionEffectiveness": detection_effectiveness,
        "detectionEffectivenessNote": detection_effectiveness_notes,
        "mitigation_uri": mitigation_uri,
        "mitigationPhase": mitigation_phases,
        "mitigationEffectiveness": mitigation_effectiveness,
        "mitigationDescription": mitigation_descriptions,
        "related_weaknesses": related
    }
    out_items.append(item)

# write JSONL
with JSONL_OUT.open("w", encoding="utf-8") as fh:
    for it in out_items:
        fh.write(json.dumps(it, ensure_ascii=False) + "\\n")

# write CSV summary (flatten some lists into semicolon-separated strings)
with CSV_OUT.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=[
        "cwe_id","title","description",
        "modeOfIntroductionPhase","modeOfIntroductionNote",
        "consequenceScope","consequenceImpact","consequenceNote",
        "detectionMethod","detectionEffectiveness",
        "mitigationPhase","mitigationEffectiveness","mitigationDescription",
        "related_weaknesses"
    ])
    writer.writeheader()
    for it in out_items:
        row = {
            "cwe_id": it["cwe_id"],
            "title": it["title"],
            "description": it["description"],
            "modeOfIntroductionPhase": ";".join(it.get("modeOfIntroductionPhase", [])),
            "modeOfIntroductionNote": ";".join(it.get("modeOfIntroductionNote", [])),
            "consequenceScope": ";".join(["|".join(s) for s in it.get("consequenceScope", [])]),
            "consequenceImpact": ";".join(["|".join(s) for s in it.get("consequenceImpact", [])]),
            "consequenceNote": ";".join([",".join(n) for n in it.get("consequenceNote", [])]),
            "detectionMethod": ";".join(it.get("detectionMethod", [])),
            "detectionEffectiveness": ";".join(it.get("detectionEffectiveness", [])),
            "mitigationPhase": ";".join(it.get("mitigationPhase", [])),
            "mitigationEffectiveness": ";".join(it.get("mitigationEffectiveness", [])),
            "mitigationDescription": ";".join(it.get("mitigationDescription", [])),
            "related_weaknesses": ";".join([f\"{r['cwe_id']}|{r['nature']}|{r['view_id']}\" for r in it.get("related_weaknesses", [])])
        }
        writer.writerow(row)

print(f"Wrote {len(out_items)} entries to:")
print(" -", JSONL_OUT)
print(" -", CSV_OUT)
